# ⚽ Real Madrid Data Collection & Storage 📊
> Pulling and storing all data needed for **Pre-Match** and **Post-Match** analysis
---

## 📚 Import Libraries & Setup

In [1]:
import requests
import pandas as pd
import json
import numpy as np
import os
from warnings import filterwarnings
filterwarnings('ignore')

api_base = r'https://football-backend-app.victoriouswater-69fff737.swedencentral.azurecontainerapps.io/'
TEAM_ID = 2829  # Real Madrid
NUM_MATCHES = 5
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

# Cached HTTP helper
_api_cache: dict = {}
_DEFAULT_TIMEOUT = 15

def cached_get(url: str, timeout: int = _DEFAULT_TIMEOUT):
    if url in _api_cache:
        return _api_cache[url]
    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code == 200:
            try:
                resp.json()
            except ValueError:
                return None
            _api_cache[url] = resp
        return resp
    except:
        return None

print("✅ Libraries loaded & cache ready")

✅ Libraries loaded & cache ready


## 1️⃣ | Fetch Team Info & Store Team ID

In [2]:
team_resp = cached_get(api_base + f'teams/{TEAM_ID}')
team_data = team_resp.json() if team_resp else {}
team_name = team_data.get('team', {}).get('name', 'Unknown')

team_info = {
    'team_id': TEAM_ID,
    'team_name': team_name,
    'raw_data': team_data
}

with open(f'{DATA_DIR}/team_info.json', 'w', encoding='utf-8') as f:
    json.dump(team_info, f, indent=2, ensure_ascii=False)

print(f"✅ Team Info saved: {team_name} (ID: {TEAM_ID})")

✅ Team Info saved: Real Madrid (ID: 2829)


## 2️⃣ | Fetch Last N Matches Info

In [3]:
def get_team_lnm(api_base, team_id, num_matchs):
    response = cached_get(api_base + f'teams/{team_id}/events/last/0')
    if response is None or response.status_code != 200:
        return 'the api is down'
    match_info = {
        match['id']: {
            'homeTeam': match['homeTeam']['name'],
            'awayTeam': match['awayTeam']['name']
        }
        for match in response.json()['events']
    }
    match_info = dict(reversed(list(match_info.items())[-num_matchs:]))
    match_info['target_team_id'] = team_id
    team_resp = cached_get(api_base + f'teams/{team_id}')
    if team_resp is None or team_resp.status_code != 200:
        return 'the api is down'
    match_info['target_team_name'] = team_resp.json().get('team').get('name')
    return match_info

teams_info = get_team_lnm(api_base, TEAM_ID, NUM_MATCHES)

# Save matches info
serializable = {}
for k, v in teams_info.items():
    serializable[str(k)] = v

with open(f'{DATA_DIR}/matches_info.json', 'w', encoding='utf-8') as f:
    json.dump(serializable, f, indent=2, ensure_ascii=False)

print(f"✅ Last {NUM_MATCHES} matches info saved")
for k, v in teams_info.items():
    if isinstance(k, int):
        print(f"   Match {k}: {v['homeTeam']} vs {v['awayTeam']}")

✅ Last 5 matches info saved
   Match 14083422: Espanyol vs Real Madrid
   Match 14083211: Real Betis vs Real Madrid
   Match 14083295: Real Madrid vs Deportivo Alavés
   Match 15632088: FC Bayern München vs Real Madrid
   Match 14083191: Real Madrid vs Girona FC


## 3️⃣ | Fetch & Store Match Statistics (Team-Level)

In [4]:
def get_match_stats(api_base, matches_info):
    all_matches_stats = []
    for match_id in matches_info.keys():
        if isinstance(match_id, int):
            if matches_info.get('target_team_name') == matches_info.get(match_id).get('awayTeam'):
                value_key = 'awayValue'
            else:
                value_key = 'homeValue'
            match_stats = {}
            match_stats['match_id'] = match_id
            match_stats['home_team'] = matches_info[match_id]['homeTeam']
            match_stats['away_team'] = matches_info[match_id]['awayTeam']
            lineups_resp = cached_get(api_base + f'events/{match_id}/lineups')
            if lineups_resp is not None and lineups_resp.status_code == 200:
                home_team = matches_info[match_id]['homeTeam']
                match_stats['team_formation'] = lineups_resp.json().get(
                    'home' if matches_info.get('target_team_name') == home_team else 'away', {}
                ).get('formation')
            else:
                match_stats['team_formation'] = None
            try:
                stats_resp = cached_get(api_base + f'events/{match_id}/statistics')
                if stats_resp is None or stats_resp.status_code != 200:
                    continue
                response = stats_resp.json()['statistics']
            except:
                continue
            for period in response:
                if period.get('period') == 'ALL':
                    response = period['groups']
            for group_stat in response:
                for stat_item in group_stat['statisticsItems']:
                    if stat_item.get('name') is not None:
                        match_stats[stat_item.get('name')] = stat_item.get(value_key)
            all_matches_stats.append(match_stats)
    return pd.DataFrame(all_matches_stats).fillna(0)

matches_stats = get_match_stats(api_base, teams_info)
matches_stats.to_csv(f'{DATA_DIR}/match_statistics.csv', index=False)

print(f"✅ Match statistics saved: {matches_stats.shape[0]} matches, {matches_stats.shape[1]} features")
matches_stats.head()

✅ Match statistics saved: 5 matches, 54 features


,match_id,home_team,away_team,team_formation,Ball possession,Expected goals,Big chances,Total shots,Goalkeeper saves,Corner kicks,...,Goals prevented,High claims,Punches,Goal kicks,Through balls,Errors lead to a goal,Big saves,Distance covered,Number of sprints,Red cards
0,14083422,Espanyol,Real Madrid,4-4-2,67,2.10,2,16,3,6,...,0.4819,2.0,0.0,12,0.0,0.0,0.0,0.00000,0.0,0.0
1,14083211,Real Betis,Real Madrid,4-4-2,48,1.19,2,12,3,6,...,0.7449,0.0,0.0,13,1.0,1.0,2.0,0.00000,0.0,0.0
2,14083295,Real Madrid,Deportivo Alavés,4-4-2,61,1.62,0,24,4,9,...,0.3237,1.0,0.0,7,1.0,0.0,1.0,0.00000,0.0,0.0
3,15632088,FC Bayern München,Real Madrid,4-4-2,31,2.25,3,12,4,2,...,-0.9168,1.0,0.0,7,1.0,0.0,3.0,105.38747,96.0,2.0
4,14083191,Real Madrid,Girona FC,4-4-2,61,2.30,2,22,1,10,...,-0.1227,0.0,0.0,9,2.0,0.0,1.0,0.00000,0.0,0.0


## 4️⃣ | Fetch & Store Player Statistics (Per Match)

In [5]:
def get_players_stats(api_base, matches_info):
    all_players_stats = []
    for match_id in matches_info.keys():
        if isinstance(match_id, int):
            is_home = matches_info.get('target_team_name') == matches_info.get(match_id).get('homeTeam')
            team_key = 'home' if is_home else 'away'
            try:
                response = cached_get(api_base + f'events/{match_id}/lineups')
                if response is None or response.status_code != 200:
                    continue
                data = response.json()
                if team_key in data and isinstance(data[team_key], dict) and 'players' in data[team_key]:
                    for player in data[team_key]['players']:
                        if not isinstance(player, dict):
                            continue
                        player_stat = {
                            'match_id': match_id,
                            'player_id': player.get('player', {}).get('id'),
                            'player_name': player.get('player', {}).get('name'),
                            'position': player.get('position'),
                            'shirt_number': player.get('shirtNumber'),
                            'substitute': player.get('substitute', False),
                            'captain': player.get('captain', False)
                        }
                        if 'statistics' in player and isinstance(player['statistics'], dict):
                            for stat_name, stat_value in player['statistics'].items():
                                if isinstance(stat_value, dict):
                                    if stat_name == 'ratingVersions':
                                        player_stat['rating_original'] = stat_value.get('original', 0)
                                        player_stat['rating_alternative'] = stat_value.get('alternative', 0)
                                    else:
                                        player_stat[stat_name] = str(stat_value)
                                else:
                                    player_stat[stat_name] = stat_value
                        all_players_stats.append(player_stat)
            except Exception as e:
                print(f'Error match {match_id}: {e}')
                continue
    if not all_players_stats:
        return pd.DataFrame()
    df = pd.DataFrame(all_players_stats).fillna(0)
    if 'ratingVersions' in df.columns:
        df = df.drop('ratingVersions', axis=1)
    return df

players_stats = get_players_stats(api_base, teams_info)
players_stats.to_csv(f'{DATA_DIR}/player_statistics.csv', index=False)

print(f"✅ Player statistics saved: {players_stats.shape[0]} rows, {players_stats.shape[1]} features")
print(f"   Unique players: {players_stats['player_name'].nunique()}")
players_stats.head()

✅ Player statistics saved: 114 rows, 85 features
   Unique players: 33


,match_id,player_id,player_name,position,shirt_number,substitute,captain,totalPass,accuratePass,totalLongBalls,...,errorLeadToAGoal,topSpeed,kilometersCovered,numberOfSprints,metersCoveredWalkingKm,metersCoveredJoggingKm,metersCoveredRunningKm,metersCoveredHighSpeedRunningKm,metersCoveredSprintingKm,clearanceOffLine
0,14083422,857574,Andriy Lunin,G,13,False,False,40.0,36.0,12.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,14083422,795064,Trent Alexander-Arnold,D,12,False,False,69.0,62.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,14083422,142622,Antonio Rüdiger,D,22,False,False,65.0,62.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,14083422,1176744,Dean Huijsen,D,24,False,False,97.0,89.0,14.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,14083422,792073,Ferland Mendy,D,23,False,False,11.0,11.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5️⃣ | Fetch & Store Player Positions (Lineup-Based)

In [6]:
def get_player_positions(api_base, matches_info):
    target_team = matches_info.get('target_team_name')
    player_agg = {}
    match_ids = [k for k in matches_info.keys() if isinstance(k, int)]
    for match_id in match_ids:
        try:
            match_meta = matches_info.get(match_id, {})
            side = 'home' if match_meta.get('homeTeam') == target_team else 'away'
            resp = cached_get(api_base + f'events/{match_id}/lineups')
            if resp is None or resp.status_code != 200:
                continue
            team_data = resp.json().get(side, {})
            for p in team_data.get('players', []):
                player = p['player']
                pid = str(player['id'])
                pname = player['name']
                prole = p.get('position', 'Unknown')
                if pid not in player_agg:
                    player_agg[pid] = {'name': pname, 'base_role': prole, 'roles_played': set(), 'match_count': 0}
                player_agg[pid]['roles_played'].add(prole)
                player_agg[pid]['match_count'] += 1
        except Exception as e:
            print(f'Error match {match_id}: {e}')

    role_mapping = {
        'G': ['G'], 'D': ['CB', 'LB', 'RB', 'LWB', 'RWB'],
        'M': ['CM', 'CDM', 'CAM', 'RM', 'LM'], 'F': ['ST', 'LW', 'RW', 'CF']
    }
    positions = []
    for pid, data in player_agg.items():
        base = data['base_role']
        mapped = role_mapping.get(base, ['CM', 'CDM', 'CAM'])
        positions.append({
            'player_id': pid, 'player_name': data['name'],
            'role': base, 'specific_role': mapped,
            'matches_played': data['match_count'],
            'roles_played': list(data['roles_played'])
        })
    return pd.DataFrame(positions)

positions_df = get_player_positions(api_base, teams_info)
positions_df.to_csv(f'{DATA_DIR}/player_positions.csv', index=False)

print(f"✅ Player positions saved: {positions_df.shape[0]} players")
positions_df.head(10)

✅ Player positions saved: 33 players


,player_id,player_name,role,specific_role,matches_played,roles_played
0,857574,Andriy Lunin,G,[G],5,[G]
1,795064,Trent Alexander-Arnold,D,"[CB, LB, RB, LWB, RWB]",4,[D]
2,142622,Antonio Rüdiger,D,"[CB, LB, RB, LWB, RWB]",4,[D]
3,1176744,Dean Huijsen,D,"[CB, LB, RB, LWB, RWB]",5,[D]
4,792073,Ferland Mendy,D,"[CB, LB, RB, LWB, RWB]",5,[D]
5,831808,Federico Valverde,M,"[CM, CDM, CAM, RM, LM]",5,[M]
6,2237795,Thiago Pitarch,M,"[CM, CDM, CAM, RM, LM]",5,[M]
7,859025,Aurélien Tchouaméni,M,"[CM, CDM, CAM, RM, LM]",3,[M]
8,868812,Vinicius Júnior,M,"[CM, CDM, CAM, RM, LM]",5,"[M, F]"
9,991011,Jude Bellingham,F,"[ST, LW, RW, CF]",5,"[M, F]"


## 6️⃣ | Fetch & Store Lineups (Raw) for Each Match

In [7]:
all_lineups = {}
match_ids = [k for k in teams_info.keys() if isinstance(k, int)]
for match_id in match_ids:
    resp = cached_get(api_base + f'events/{match_id}/lineups')
    if resp and resp.status_code == 200:
        all_lineups[str(match_id)] = resp.json()
        print(f"   ✅ Lineups fetched for match {match_id}")

with open(f'{DATA_DIR}/lineups_raw.json', 'w', encoding='utf-8') as f:
    json.dump(all_lineups, f, indent=2, ensure_ascii=False)

print(f"✅ Raw lineups saved for {len(all_lineups)} matches")

   ✅ Lineups fetched for match 14083422
   ✅ Lineups fetched for match 14083211
   ✅ Lineups fetched for match 14083295
   ✅ Lineups fetched for match 15632088
   ✅ Lineups fetched for match 14083191
✅ Raw lineups saved for 5 matches


## 7️⃣ | Fetch & Store Raw Statistics JSON for Each Match

In [8]:
all_raw_stats = {}
for match_id in match_ids:
    resp = cached_get(api_base + f'events/{match_id}/statistics')
    if resp and resp.status_code == 200:
        all_raw_stats[str(match_id)] = resp.json()
        print(f"   ✅ Statistics fetched for match {match_id}")

with open(f'{DATA_DIR}/statistics_raw.json', 'w', encoding='utf-8') as f:
    json.dump(all_raw_stats, f, indent=2, ensure_ascii=False)

print(f"✅ Raw statistics saved for {len(all_raw_stats)} matches")

   ✅ Statistics fetched for match 14083422
   ✅ Statistics fetched for match 14083211
   ✅ Statistics fetched for match 14083295
   ✅ Statistics fetched for match 15632088
   ✅ Statistics fetched for match 14083191
✅ Raw statistics saved for 5 matches


## 8️⃣ | Fetch & Store Fatigue Data (Post-Match)

In [9]:
def get_fatigue(api_base, team_id):
    players_details = []
    matches_info = get_team_lnm(api_base, team_id, 1)
    for match_id in matches_info:
        if isinstance(match_id, int):
            is_home = matches_info.get('target_team_name') == matches_info.get(match_id).get('homeTeam')
            team_key = 'home' if is_home else 'away'
            try:
                data = cached_get(api_base + f'events/{match_id}/lineups')
                if data is None:
                    continue
                data = data.json()
                if team_key in data and 'players' in data[team_key]:
                    for player in data[team_key]['players']:
                        if not isinstance(player, dict):
                            continue
                        players_details.append({
                            'player_id': str(player.get('player', {}).get('id')),
                            'name': player.get('player', {}).get('name'),
                            'position': player.get('position'),
                            'minutes_played': player.get('statistics', {}).get('minutesPlayed', 0),
                        })
            except Exception as e:
                print(f"Error: {e}")
    if not players_details:
        return {}
    df = pd.DataFrame(players_details)
    min_s, max_s = df['minutes_played'].min(), df['minutes_played'].max()
    df['fatigue_index'] = round(100 * (df['minutes_played'] - min_s) / (max_s - min_s) if max_s != min_s else 0)
    df['injury_risk_level'] = df.apply(
        lambda r: 'High' if r['minutes_played'] >= 180 else (
            'Low' if r['fatigue_index'] < 60 else ('Moderate' if r['fatigue_index'] < 80 else 'High')
        ), axis=1
    )
    return df

fatigue_df = get_fatigue(api_base, TEAM_ID)
if isinstance(fatigue_df, pd.DataFrame) and not fatigue_df.empty:
    fatigue_df.to_csv(f'{DATA_DIR}/fatigue_data.csv', index=False)
    print(f"✅ Fatigue data saved: {fatigue_df.shape[0]} players")
    fatigue_df.head()
else:
    print("⚠️ Could not fetch fatigue data")

✅ Fatigue data saved: 22 players


## 9️⃣ | Summary of All Stored Data

In [10]:
print("=" * 60)
print(f"📦 DATA COLLECTION COMPLETE — Real Madrid (ID: {TEAM_ID})")
print("=" * 60)
print(f"\n📁 All files stored in: ./{DATA_DIR}/\n")

for f_name in sorted(os.listdir(DATA_DIR)):
    f_path = os.path.join(DATA_DIR, f_name)
    size_kb = os.path.getsize(f_path) / 1024
    print(f"   📄 {f_name:30s}  ({size_kb:.1f} KB)")

print(f"\n{'=' * 60}")
print("✅ Ready for Pre-Match and Post-Match analysis!")

📦 DATA COLLECTION COMPLETE — Real Madrid (ID: 2829)

📁 All files stored in: ./data/

   📄 fatigue_data.csv                (0.9 KB)
   📄 lineups_raw.json                (642.8 KB)
   📄 match_statistics.csv            (1.7 KB)
   📄 matches_info.json               (0.5 KB)
   📄 player_positions.csv            (2.2 KB)
   📄 player_statistics.csv           (51.4 KB)
   📄 statistics_raw.json             (268.3 KB)
   📄 team_info.json                  (8.4 KB)

✅ Ready for Pre-Match and Post-Match analysis!
